# Lesson 6 — Class imbalance and cost-sensitive training

Lesson 6 — Class imbalance and cost-sensitive training
======================================================

GURU NOTE
---------
Real traffic is ~99% benign. Train naively on that and the model discovers a lazy
shortcut: "always say benign" scores 99% accuracy while catching ZERO attacks.
Accuracy is a liar on imbalanced data. This lesson shows the trap and the two-line
fix your repo uses.

TWO IDEAS, both living in one line of train.py (line 41):

  1. CLASS BALANCING — up-weight the rare class so its examples count as much as
     the common one. Now the model can't ignore attacks.

  2. FALSE-POSITIVE COST (`fp_weight`) — for a firewall, blocking a real customer
     is worse than logging one attack. So we make benign mistakes EXTRA expensive
     (fp_weight=6 in the repo: a false alarm costs 6x a miss). This trades a little
     recall for a lot of precision — exactly what a WAF wants.

The one line in the repo:
    sample_w = np.where(y >= 0.5, n/(2*n_pos), fp * n/(2*n_neg))
We build it and feel each half.

Run me:  python lessons/lesson_06_class_imbalance.py

## Setup

These notebooks *import* the real code from `lessons/*.py` — one source of truth, no forked copies. This cell finds that folder.

In [1]:
import sys, pathlib
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / 'lessons' / '_util.py').exists():
        sys.path.insert(0, str(base / 'lessons'))
        print('lessons on path:', base / 'lessons'); break
else:
    raise RuntimeError('Open this notebook inside the xorguard-ml-course folder.')

lessons on path: /Users/anil/Documents/developer/vscodeworkspaces/xordev/xorguard-ml-course/lessons


## The key code (shown live from the source file)

Read it, then copy any function into a new cell and tweak it — the originals live in `lessons/lesson_06_class_imbalance.py`.

In [2]:
import inspect
from lesson_06_class_imbalance import fit, report
for fn in (fit, report):
    print(inspect.getsource(fn)); print('-' * 72)

def fit(X, y, fp_weight: float = 1.0, balance: bool = True,
        l2: float = 0.05, steps: int = 800, lr: float = 0.4):
    """Logistic regression with the repo's per-sample class weights.

    balance=False -> every example counts equally (the naive trap).
    balance=True  -> rare class up-weighted; fp_weight makes benign mistakes cost
                     more. This is train.fit_logistic's sample_w, exactly.
    """
    n, d = X.shape
    w = np.zeros(d)
    w[-1] = -0.5
    n_pos = max(float(np.sum(y >= 0.5)), 1.0)
    n_neg = max(float(np.sum(y < 0.5)), 1.0)
    if balance:
        sample_w = np.where(y >= 0.5,
                            n / (2.0 * n_pos),
                            max(fp_weight, 1.0) * n / (2.0 * n_neg))
    else:
        sample_w = np.ones(n)
    for step_i in range(steps):
        p = sigmoid(X @ w)
        err = (p - y) * sample_w          # <-- the only change from Lesson 4
        grad = X.T @ err / n
        step_lr = lr / (1.0 + step_i / 250.0)
      

## Run the full narrated lesson

This executes the same walkthrough as `python lessons/lesson_06_class_imbalance.py`, printing each step with its result.

In [3]:
from lesson_06_class_imbalance import main
main()


══════════════════════════════════════════════════════
Lesson 6 — Class imbalance and cost-sensitive training
══════════════════════════════════════════════════════
    dataset: 1900 benign vs 100 attacks (5% positive). This is a MILD version of real traffic.

▸ The trap: train with every example weighted equally.
    naive (no balancing)
      accuracy=95.0%  recall(caught attacks)=0.0%  precision=0.0%  TNR(benign spared)=100.0%
      confusion: TP=0 FP=0 FN=100 TN=1900
    Look at accuracy vs recall. High accuracy can hide a model that
    misses most attacks, because benign examples dominate the loss.

▸ Fix #1: balance the classes so attacks count as much as benign.
    balanced (fp_weight=1)
      accuracy=98.9%  recall(caught attacks)=99.0%  precision=82.5%  TNR(benign spared)=98.9%
      confusion: TP=99 FP=21 FN=1 TN=1879
    Recall jumps: the rare class now pulls the weights toward catching it.

▸ Fix #2: make FALSE ALARMS expensive (fp_weight), like a real WAF.
    A firewal

## Exercises

Try these in new cells below. Worked answers are in `solutions/lesson_06_solution.py`.

```
1. Make the data 99% benign (9900 vs 100). Re-run the naive model. What is its
   accuracy, and its recall? Explain in one sentence why accuracy is useless
   here.

2. Find the fp_weight (to the nearest integer) where TNR first reaches 99% on
   this data. What did recall drop to in exchange? This is the precision/recall
   trade-off you'll formalize with thresholds in Lesson 7.

3. Set fp_weight very high (say 100). What happens to recall, and why would a
   security team NOT want to ship that model even though it never false-alarms?
```

In [4]:
# your work here
